# Credit Default Risk — Probability-of-Default Model

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["MLFLOW_TRACKING_URI"] = UserSecretsClient().get_secret("MLFLOW_TRACKING_URI").strip()

!git clone https://github.com/MeetRVyas/vouch-credit-risk.git
%cd vouch-credit-risk
!pip install -q ".[ml]"

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

# make the repo's src/ package importable, whether this notebook is run
# from the repo root (local/CI) or after uploading the repo as a Kaggle
# Dataset/Utility Script attached to the competition notebook.
for candidate in ["src", "../src", "/kaggle/input/credit-risk-src/src"]:
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)


pl.Config.set_tbl_rows(10)
np.random.seed(42)


## 1. Load data

Prefers the real Kaggle competition files; falls back to the schema-matched synthetic sample otherwise.

In [ ]:
KAGGLE_DIR = "/kaggle/input/competitions/home-credit-default-risk"

if os.path.exists(os.path.join(KAGGLE_DIR, "application_train.csv")):
    APPLICATION_PATH = os.path.join(KAGGLE_DIR, "application_train.csv")
    BUREAU_PATH = os.path.join(KAGGLE_DIR, "bureau.csv")
    RUNNING_ON_KAGGLE = True
else:
    from credit_risk.data.synthetic import write_sample_csvs

    SAMPLE_DIR = "data/sample"
    if not os.path.exists(os.path.join(SAMPLE_DIR, "application_train_sample.csv")):
        write_sample_csvs(SAMPLE_DIR, n_applicants=2000)
    APPLICATION_PATH = os.path.join(SAMPLE_DIR, "application_train_sample.csv")
    BUREAU_PATH = os.path.join(SAMPLE_DIR, "bureau_sample.csv")
    RUNNING_ON_KAGGLE = False

print(f"running on Kaggle: {RUNNING_ON_KAGGLE}")
print(f"application_train: {APPLICATION_PATH}")
print(f"bureau:             {BUREAU_PATH}")


In [ ]:
# N_TRIALS = 2
N_TRIALS = 50
DEVICE = "cuda" if RUNNING_ON_KAGGLE else "cpu"
# NFOLDS = 3
NFOLDS = 5
# N_ESTIMATORS = 30
N_ESTIMATORS = 200
# EXPERIMENT_NAME = "credit-default-risk"
EXPERIMENT_NAME = "credit-default-risk-final"

## 2. Temporal leakage check — verify personally

This is the one check called out as too dangerous to leave to AI-generated
code: confirm nothing derived from `bureau` postdates the loan application
it's joined onto. `credit_risk.data.duckdb_pipeline.load_and_join` already
runs this as a hard gate before aggregating (it will raise if it fails) —
it's re-run explicitly here, separately, so the result is visible and
inspectable rather than buried inside a pipeline call.

In [ ]:
import duckdb

from credit_risk.data.leakage_checks import check_bureau_temporal_integrity

_con = duckdb.connect(":memory:")
_raw_bureau = _con.execute(f"select * from read_csv_auto('{BUREAU_PATH}')").pl()
_con.close()

report = check_bureau_temporal_integrity(_raw_bureau)
print(report.summary())
assert report.passed_hard_check, "Do not proceed past this cell until this passes."


## 3. Bureau aggregation (DuckDB) + join, feature engineering (Polars)

In [ ]:
from credit_risk.data.duckdb_pipeline import load_and_join
from credit_risk.features.engineering import build_feature_frame, get_feature_columns

joined = load_and_join(APPLICATION_PATH, BUREAU_PATH)
print("joined shape:", joined.shape)

feat = build_feature_frame(joined)
feature_cols = get_feature_columns(feat)
print("feature frame shape:", feat.shape, "| n_features:", len(feature_cols))

feat.select(
    ["SK_ID_CURR", "TARGET", "credit_to_income_ratio", "annuity_to_income_ratio", "bureau_debt_to_credit_ratio"]
).head()


## 4. Class balance

In [ ]:
X = feat.select(feature_cols)
y = feat["TARGET"].to_numpy()

default_rate = y.mean()
print(f"n={len(y):,}  default rate={default_rate:.3%}")

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["repaid", "default"], [1 - default_rate, default_rate], color=["#2F6F4E", "#8C3B33"])
ax.set_ylabel("share of applicants")
ax.set_title("Target balance")
plt.show()


## 5. MLflow tracking

Points at the self-hosted server (`infra/mlflow-server/`) via
`MLFLOW_TRACKING_URI` — set as a **Kaggle Secret** on the real run, never
hardcoded (see `infra/mlflow-server/README.md` for the Cloudflare Tunnel
setup). Falls back to a local SQLite store here so this cell still runs
without a live server (MLflow's plain filesystem backend is in maintenance
mode as of MLflow 3.x — a DB-backed store is the current recommendation
even for a local fallback).

`autolog()` is scoped to the final single-shot fit below rather than turned
on globally: our CV/tuning cells fit several models (one per fold) inside
one logical MLflow run each, and autolog tries to attach every fit's params
to that same run -- which conflicts the moment `scale_pos_weight` (computed
per-fold) differs between folds. Explicit `log_params`/`log_metrics` for the
CV/tuning runs avoids that; autolog is the right tool for the one-shot final
fit, where it cleanly captures the training curve, feature importance, and
model signature.

In [ ]:
import mlflow

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "sqlite:///mlruns.db")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI:", tracking_uri)


## 6. Baseline stratified CV

Never a single split — target is ~8% positive.

In [ ]:
from credit_risk.models.train import XGBParams, run_stratified_cv

# device="cpu" here (no GPU in this environment); on Kaggle: XGBParams(device="cuda")
baseline_params = XGBParams(device="cuda", n_estimators=N_ESTIMATORS)
# baseline_params = XGBParams(device="cuda", n_estimators=200)

with mlflow.start_run(run_name="baseline_cv"):
    baseline_result = run_stratified_cv(X, y, baseline_params, n_splits=5)
    mlflow.log_params(baseline_params.__dict__)
    mlflow.log_metrics(baseline_result.mean_metrics)

for k, v in baseline_result.mean_metrics.items():
    print(f"{k:12s} {v:.4f}")


## 7. Hyperparameter tuning (Optuna)

Bayesian search (TPE, Optuna's default sampler) under a limited compute
budget. **`N_TRIALS` below is a small demo budget** for this environment —
the real run on Kaggle GPU should use a larger budget (e.g. 50–100+ trials);
every trial is still logged to MLflow as a nested run either way.

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial: optuna.Trial) -> float:
    from credit_risk.models.train import XGBParams, run_stratified_cv

    params = XGBParams(
        device=DEVICE,
        n_estimators=trial.suggest_int("n_estimators", 150, 600, step=50),
        max_depth=trial.suggest_int("max_depth", 3, 9),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 10.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
    )
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        result = run_stratified_cv(X, y, params, n_splits=3)  # 3 folds inside the search loop, 5 for the final CV
        mlflow.log_params(trial.params)
        mlflow.log_metrics(result.mean_metrics)
    return result.mean_metrics["roc_auc"]


with mlflow.start_run(run_name="optuna_search"):
    study = optuna.create_study(direction="maximize", study_name="credit-default-risk")
    study.optimize(objective, n_trials=N_TRIALS)
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_roc_auc", study.best_value)

print("best ROC-AUC:", round(study.best_value, 4))
print("best params:", study.best_params)


## 8. Final CV with tuned hyperparameters (5-fold, held-out metrics)

Re-run the full 5-fold CV with the tuned params to get honest, held-out (out-of-fold) predictions for evaluation below — not in-sample numbers from a model that has already seen every row.

In [ ]:
from credit_risk.models.train import XGBParams

best_params = XGBParams(device=DEVICE, **study.best_params)

with mlflow.start_run(run_name="final_cv_tuned"):
    final_cv_result = run_stratified_cv(X, y, best_params, n_splits=NFOLDS)
    mlflow.log_params(best_params.__dict__)
    mlflow.log_metrics(final_cv_result.mean_metrics)

oof_predictions = final_cv_result.oof_predictions
for k, v in final_cv_result.mean_metrics.items():
    print(f"{k:12s} {v:.4f}")


## 9. Evaluation — credit-scoring metrics on held-out (OOF) predictions

In [ ]:
from credit_risk.evaluation.metrics import compute_classification_metrics

metrics = compute_classification_metrics(y, oof_predictions)
pd.DataFrame([metrics.as_dict()]).T.rename(columns={0: "value"}).style.format("{:.4f}")


## 10. Calibration — reliability curve

Raw PD often feeds pricing/risk bands downstream, not just ranking, so calibration matters here, not just discrimination (AUC/Gini/KS).

In [ ]:
from credit_risk.evaluation.metrics import reliability_curve

mean_pred, mean_obs, counts = reliability_curve(y, oof_predictions, n_bins=10)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], linestyle="--", color="#8b9690", label="perfectly calibrated")
ax.plot(mean_pred, mean_obs, marker="o", color="#A6742C", label="model")
ax.set_xlabel("mean predicted probability")
ax.set_ylabel("observed default rate")
ax.set_title("Reliability curve")
ax.legend()
plt.show()


## 11. Business framing — default rate by approval-rate cutoff

The threshold-dependent complement to AUC: what happens at different approval rates, not just a threshold-free ranking metric.

In [ ]:
from credit_risk.evaluation.metrics import approval_cutoff_table

cutoffs = approval_cutoff_table(y, oof_predictions)
pd.DataFrame(cutoffs).style.format({
    "approval_rate": "{:.0%}",
    "score_cutoff": "{:.4f}",
    "default_rate_among_approved": "{:.3%}",
    "overall_default_rate": "{:.3%}",
})


## 12. Final model fit (full training data) + SHAP + save

One fit, one run: `autolog()` is enabled just for this call (see sec. 5) so it captures the training curve, feature importance, and model signature automatically, alongside our explicit credit-scoring metrics, the SHAP artifact, and the local model artifact the API loads.

In [ ]:
from credit_risk.explain.shap_explain import build_explainer, compute_shap_values, save_summary_plot
from credit_risk.models.artifact import ModelArtifact
from credit_risk.models.train import train_final_model

MODEL_VERSION = "v1.0.0-" + ("kaggle" if RUNNING_ON_KAGGLE else "local-sample")

mlflow.xgboost.autolog(log_models=True)

with mlflow.start_run(run_name="final_model"):
    final_model = train_final_model(X, y, best_params)

    # custom credit-scoring metrics (Gini/KS/Brier) aren't part of autolog's
    # standard capture -- log them explicitly alongside it
    mlflow.log_metrics(metrics.as_dict())

    explainer = build_explainer(final_model)
    X_pd = X.to_pandas()
    shap_sample = X_pd.sample(min(1000, len(X_pd)), random_state=42)
    shap_values = compute_shap_values(explainer, shap_sample)

    os.makedirs("artifacts/model", exist_ok=True)
    summary_plot_path = save_summary_plot(shap_values, shap_sample, "artifacts/model/shap_summary.png")
    mlflow.log_artifact(summary_plot_path)

    artifact = ModelArtifact(model=final_model, feature_columns=feature_cols, model_version=MODEL_VERSION)
    artifact.save("artifacts/model")

mlflow.xgboost.autolog(disable=True)
print(f"saved model artifact, version={MODEL_VERSION}")
print("saved:", summary_plot_path)


In [ ]:
from IPython.display import Image

Image(filename="artifacts/model/shap_summary.png")


## 13. Scope notes

> **On these numbers:** the metrics above (ROC-AUC ≈ 0.94) are from the
> synthetic sample, which has a stronger, more deterministic
> signal-to-noise ratio than real applicant data by construction (see
> `credit_risk.data.synthetic`) — they confirm the pipeline and metrics are
> computed correctly, not what to expect from the real dataset. The
> **realistic target on the real two-table dataset is ~0.74–0.78 ROC-AUC**
> (see top of notebook).

- **Out of scope (deliberate):** model registry, staging→prod promotion —
  no real second environment exists to justify them.
- **Two-table scope:** `application_train` + `bureau` only. The full
  7-table dataset (`bureau_balance`, `previous_application`,
  `POS_CASH_balance`, `credit_card_balance`, `installments_payments`) can
  push ROC-AUC past 0.80 — not chased here; CV/metric rigor mattered more
  than the third decimal at this scope.
- **Known dataset caveat (see `credit_risk/data/leakage_checks.py`):**
  `bureau.csv` is extracted at one fixed date for the whole table, not per
  application, so `CREDIT_ACTIVE`/`AMT_CREDIT_SUM_DEBT` reflect status as of
  that extraction, not each applicant's own application date. Flagged, not
  hidden — see the module docstring for detail.
- **Next step:** the `mlruns/` produced against the self-hosted server (not
  this notebook's local fallback store), the real SHAP plot, and the live
  FastAPI endpoint are produced from *this real run on Kaggle*, not from
  the synthetic sample used to make sure every cell above actually runs.